# Mansoura Mobility: promotion risk investigation

Mansoura Mobility is considering a passenger discount campaign that could increase ride requests by 20–30% during selected hours.

This analysis examines offer acceptance, ride completion, cancellations, ratings, and reports across pickup zones and time blocks. It compares weaker segments with the marketplace and finishes with a ride-level extract for reviewing El Mokhtalat during the evening.

The records are synthetic. This notebook explores operational risk; it does not establish campaign impact or give a final launch recommendation.

## Analysis approach

PostgreSQL queries extract the data, summarize related payment and report records, and return one row per offer. Python is used for segment summaries, comparisons, and conditional formatting.

The analysis uses parameterized queries, Cairo-local timestamps, row-grain checks, and a filtered investigation extract.

Stage 3 has two known metric-definition mismatches: its payment-secured count includes completed rides only, and its report-rate denominator uses completed rides. Stage 5 separately calculates the report rate using started rides. Stage 3 figures are exploratory and should not be presented as the defined business KPIs.

## Scope

The investigation covers the final three months of activity, comparing pickup-zone and time-block segments. It focuses on observed patterns and cases requiring review, rather than claiming what caused the differences.

## Connection and notebook helpers


In [72]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

project_root = Path(r"C:\Python\projects\pgSQL")
load_dotenv(project_root / ".env", override=True)
sys.path.insert(0, str(project_root / "src"))

from MAna.database import connect_postgres, execute_query

db = connect_postgres(driver="psycopg2")


def run_sql(query, params=None):
    if not query.strip():
        return pd.DataFrame({"status": ["Query not written yet."]})
    return execute_query(query, db, params=params, return_results=True)


def explain_sql(query, params=None):
    if not query.strip():
        return pd.DataFrame({"status": ["Query not written yet."]})
    statement = "EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)\n" + query
    return execute_query(statement, db, params=params, return_results=True)

[OK] Connected to database: postgresql+psycopg2://localhost/mansoura_mobility


## Analysis settings

The analysis period covers the final three months of the dataset. The end timestamp is exclusive. Query values are supplied through `analysis_params`.

In [73]:
analysis_params = {
    "analysis_start": "2026-06-01T00:00:00+03:00",
    "analysis_end": "2026-09-01T00:00:00+03:00",
    "minimum_segment_offers": 75,
    "minimum_started_rides": 30,
}

analysis_params

{'analysis_start': '2026-06-01T00:00:00+03:00',
 'analysis_end': '2026-09-01T00:00:00+03:00',
 'minimum_segment_offers': 75,
 'minimum_started_rides': 30}

## Business definitions

Metric definitions:

- **Offer:** one request sent to one driver.
- **Accepted offer:** `offers.offer_status = 'ACCEPTED'`.
- **Payment secured:** the ride has at least one payment attempt with status `AUTHORIZED` or `CAPTURED`.
- **Started ride:** status is `IN_PROGRESS`, `COMPLETED`, or `TERMINATED_EARLY`.
- **Completed ride:** status is `COMPLETED`.
- **Offer acceptance rate:** accepted offers divided by all offers.
- **Completion rate:** completed rides divided by accepted offers.
- **Payment failure rate:** rides with payment attempts but no successful attempt, divided by rides with at least one attempt.
- **Report rate:** rides with at least one report per 1,000 started rides.


## Stage 1 — Initial data checks

Inspect the offers and check for accepted offers without a linked ride.

In [74]:
sql = """
SELECT * FROM offers;
"""
run_sql(sql)

,offer_id,passenger_id,driver_id,vehicle_id,pickup_zone_id,dropoff_zone_id,initial_fare_egp,offer_status,initiated_at,declined_by,decided_at,decline_reason
0,1,972,1023,56,7,1,50.0,ACCEPTED,2026-03-03 23:23:37+00:00,NaN,2026-03-03 23:25:30+00:00,NaN
1,2,27,1058,102,14,13,35.0,ACCEPTED,2026-03-04 00:46:50+00:00,NaN,2026-03-04 00:49:51+00:00,NaN
2,3,439,1078,125,14,4,50.0,ACCEPTED,2026-03-04 01:53:22+00:00,NaN,2026-03-04 01:56:33+00:00,NaN
3,4,748,989,11,1,12,50.0,ACCEPTED,2026-03-04 03:42:34+00:00,NaN,2026-03-04 03:44:42+00:00,NaN
4,5,535,1017,49,7,3,45.0,ACCEPTED,2026-03-04 04:04:26+00:00,NaN,2026-03-04 04:08:51+00:00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
13995,13996,115,1006,33,6,13,80.0,ACCEPTED,2026-08-30 18:17:18+00:00,NaN,2026-08-30 18:18:24+00:00,NaN
13996,13997,603,1008,35,12,3,55.0,ACCEPTED,2026-08-30 18:23:07+00:00,NaN,2026-08-30 18:24:48+00:00,NaN
13997,13998,557,1068,113,1,14,60.0,WITHDRAWN,2026-08-30 18:58:20+00:00,PASSENGER,2026-08-30 19:01:06+00:00,PRICE_DISAGREEMENT
13998,13999,720,1019,51,11,4,45.0,ACCEPTED,2026-08-30 19:10:49+00:00,NaN,2026-08-30 19:12:23+00:00,NaN


In [75]:
quality_check_sql = """
SELECT 'accepted offers without a ride' as check_name, COUNT(*) as violation_count
FROM offers
LEFT JOIN rides
ON offers.offer_id = rides.offer_id
WHERE offers.offer_status = 'ACCEPTED' AND rides.offer_id IS NULL; 
"""

run_sql(quality_check_sql, analysis_params)


,check_name,violation_count
0,accepted offers without a ride,0


### Initial inspection

The results below provide an initial check, not a complete audit of every database rule.

## Stage 2 — Build the analysis dataset

The SQL extract returns one row per offer within the selected period, including Cairo-local timestamps, route names, vehicle details, ride outcomes, payments, refunds, ratings, and reports.

Payment attempts, refunds, and reports are summarized before joining them to offers to avoid multiplying rows.

In [76]:
# select two columns, a column for table name, and another for column names per table
schema_sql = """
SELECT table_name, column_name
FROM information_schema.columns
WHERE table_schema = 'public'
"""
run_sql(schema_sql, analysis_params).groupby("table_name").agg(list).reset_index().to_clipboard(index=False)

accounts -> ['account_id', 'signup_at', 'last_seen_at', 'full_name', 'phone_number', 'email', 'account_status']

driver_passenger_ratings -> ['ride_id', 'attitude_score', 'punctuality_score', 'pickup_cooperation_score', 'respect_safety_score', 'submitted_at', 'comment']

drivers -> ['driver_id', 'is_accepting_offers', 'license_number']

offers -> ['offer_id', 'passenger_id', 'driver_id', 'vehicle_id', 'pickup_zone_id', 'dropoff_zone_id', 'initial_fare_egp', 'initiated_at', 'decided_at', 'offer_status', 'declined_by', 'decline_reason']

passenger_driver_ratings -> ['ride_id', 'attitude_score', 'driving_safety_score', 'vehicle_cleanliness_score', 'comfort_score', 'route_quality_score', 'submitted_at', 'comment']

passengers -> ['passenger_id']

payment_attempts -> ['captured_at', 'released_at', 'payment_attempt_id', 'ride_id', 'attempt_number', 'requested_amount_egp', 'authorized_amount_egp', 'captured_amount_egp', 'initiated_at', 'authorized_at', 'payment_status', 'provider_reference', 'failure_reason', 'payment_method']

refunds -> ['refund_id', 'payment_attempt_id', 'refund_amount_egp', 'requested_at', 'completed_at', 'refund_reason', 'refund_status', 'failure_reason']

reports -> ['report_id', 'ride_id', 'opened_at', 'resolved_at', 'report_category', 'other_report_category', 'description', 'report_status', 'resolution_notes', 'opened_by']

rides -> ['ride_id', 'offer_id', 'final_fare_egp', 'started_at', 'start_distance_metres', 'completed_at', 'ride_status', 'cancelled_by', 'cancellation_reason', 'cancellation_stage']

vehicles -> ['vehicle_id', 'driver_id', 'model_year', 'registered_at', 'retired_at', 'brand', 'model', 'vehicle_category', 'plate_number', 'service_status']

zone_routes -> ['origin_zone_id', 'destination_zone_id', 'baseline_distance_km', 'baseline_duration_minutes']

zones -> ['zone_id', 'center_latitude', 'center_longitude', 'city_name', 'zone_type', 'zone_name']


In [77]:
payments = """
SELECT * FROM payment_attempts;
"""
run_sql(payments).payment_status.value_counts()

payment_status
CAPTURED     10159
FAILED        1659
RELEASED       457
INITIATED        1
Name: count, dtype: int64

In [78]:
analysis_grain_sql = """
WITH payment_summary AS (
    SELECT
        pa.ride_id,
        COUNT(*) AS payment_attempts,
        MAX(
            CASE
                WHEN pa.payment_status IN ('AUTHORIZED', 'CAPTURED') THEN 1
                ELSE 0
            END
        ) AS ride_level_successful_payments
    FROM payment_attempts AS pa
    GROUP BY pa.ride_id
),

refund_summary AS (
    SELECT
        pa.ride_id,
        SUM(rfnd.refund_amount_egp) FILTER (
            WHERE rfnd.refund_status = 'COMPLETED'
        ) AS total_refund_amount_egp
    FROM refunds AS rfnd
    JOIN payment_attempts AS pa ON rfnd.payment_attempt_id = pa.payment_attempt_id
    GROUP BY pa.ride_id
),

report_summary AS (
    SELECT
        rprt.ride_id,
        COUNT(rprt.report_id) AS total_reports
    FROM reports AS rprt
    GROUP BY rprt.ride_id
)

SELECT
    o.offer_id,
    o.initiated_at AT TIME ZONE 'Africa/Cairo' AS cairo_time,
    EXTRACT(YEAR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS year,
    EXTRACT(MONTH FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS month,
    EXTRACT(DAY FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS day,
    EXTRACT(ISODOW FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS weekday,
    EXTRACT(HOUR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS hour,
    o.passenger_id,
    o.driver_id,
    o.vehicle_id,
    v.vehicle_category,
    zp.zone_name AS pickup_zone_name,
    zd.zone_name AS dropoff_zone_name,
    o.initial_fare_egp,
    o.offer_status,
    r.ride_id,
    r.ride_status,
    r.final_fare_egp,
    COALESCE(ps.payment_attempts, 0) AS payment_attempts,
    COALESCE(ps.ride_level_successful_payments, 0) AS ride_level_successful_payments,
    COALESCE(rs.total_refund_amount_egp, 0) AS total_refund_amount_egp,
    (
        pdr.attitude_score
        + pdr.driving_safety_score
        + pdr.vehicle_cleanliness_score
        + pdr.comfort_score
        + pdr.route_quality_score
    ) / 5.0 AS avg_passenger_to_driver_rating,
    (
        dpr.attitude_score
        + dpr.punctuality_score
        + dpr.pickup_cooperation_score
        + dpr.respect_safety_score
    ) / 4.0 AS avg_driver_to_passenger_rating,
    COALESCE(rps.total_reports, 0) AS total_reports
FROM offers AS o
LEFT JOIN rides AS r ON o.offer_id = r.offer_id
LEFT JOIN vehicles AS v ON o.vehicle_id = v.vehicle_id
LEFT JOIN zones AS zp ON o.pickup_zone_id = zp.zone_id
LEFT JOIN zones AS zd ON o.dropoff_zone_id = zd.zone_id
LEFT JOIN payment_summary AS ps ON r.ride_id = ps.ride_id
LEFT JOIN refund_summary AS rs ON r.ride_id = rs.ride_id
LEFT JOIN passenger_driver_ratings AS pdr ON r.ride_id = pdr.ride_id
LEFT JOIN driver_passenger_ratings AS dpr ON r.ride_id = dpr.ride_id
LEFT JOIN report_summary AS rps ON r.ride_id = rps.ride_id
WHERE o.initiated_at >= :analysis_start
    AND o.initiated_at < :analysis_end
ORDER BY o.initiated_at
"""


offers_table = run_sql(analysis_grain_sql, analysis_params)
offers_table

,offer_id,cairo_time,year,month,day,weekday,hour,passenger_id,driver_id,vehicle_id,...,offer_status,ride_id,ride_status,final_fare_egp,payment_attempts,ride_level_successful_payments,total_refund_amount_egp,avg_passenger_to_driver_rating,avg_driver_to_passenger_rating,total_reports
0,6315,2026-06-01 03:15:52,2026.0,6.0,1.0,1.0,3.0,56,1048,89,...,ACCEPTED,4766.0,COMPLETED,61.349998,1,1,0.0,4.6,3.75,0
1,6316,2026-06-01 03:19:39,2026.0,6.0,1.0,1.0,3.0,913,1033,70,...,AUTO_CANCELLED,NaN,NaN,NaN,0,0,0.0,NaN,NaN,0
2,6317,2026-06-01 04:37:46,2026.0,6.0,1.0,1.0,4.0,794,1026,60,...,DECLINED,NaN,NaN,NaN,0,0,0.0,NaN,NaN,0
3,6318,2026-06-01 05:24:52,2026.0,6.0,1.0,1.0,5.0,439,1057,101,...,ACCEPTED,4767.0,COMPLETED,40.080002,1,1,0.0,4.4,5.00,0
4,6319,2026-06-01 06:01:39,2026.0,6.0,1.0,1.0,6.0,771,1015,45,...,ACCEPTED,4768.0,COMPLETED,45.230000,1,1,0.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7681,13996,2026-08-30 20:17:18,2026.0,8.0,30.0,7.0,20.0,115,1006,33,...,ACCEPTED,10614.0,COMPLETED,81.510002,1,1,0.0,4.6,5.00,0
7682,13997,2026-08-30 20:23:07,2026.0,8.0,30.0,7.0,20.0,603,1008,35,...,ACCEPTED,10615.0,COMPLETED,58.880001,1,1,0.0,NaN,4.25,0
7683,13998,2026-08-30 20:58:20,2026.0,8.0,30.0,7.0,20.0,557,1068,113,...,WITHDRAWN,NaN,NaN,NaN,0,0,0.0,NaN,NaN,0
7684,13999,2026-08-30 21:10:49,2026.0,8.0,30.0,7.0,21.0,720,1019,51,...,ACCEPTED,10616.0,CANCELLED_BEFORE_START,0.000000,1,0,0.0,NaN,NaN,1


### Row-grain checks

Check duplicate offer IDs, compare the extract size with the source offer count, and inspect outcome counts.

In [79]:
offers_table.offer_id.duplicated().sum()

0

In [80]:
len(offers_table)

7686

In [81]:
grain_validation_sql = """
SELECT count(*) as number_of_offers
FROM offers AS o
WHERE o.initiated_at >= :analysis_start
    AND o.initiated_at < :analysis_end
"""

run_sql(grain_validation_sql, analysis_params)


,number_of_offers
0,7686


In [82]:
offers_table.offer_status.value_counts()

offer_status
ACCEPTED          5852
DECLINED          1050
WITHDRAWN          431
AUTO_CANCELLED     353
Name: count, dtype: int64

In [83]:
offer_status_counts = """
SELECT offer_status, COUNT(*) AS count
FROM offers
WHERE offers.initiated_at >= :analysis_start
    AND offers.initiated_at < :analysis_end
GROUP BY offer_status
ORDER BY count DESC
"""

run_sql(offer_status_counts, analysis_params)

,offer_status,count
0,ACCEPTED,5852
1,DECLINED,1050
2,WITHDRAWN,431
3,AUTO_CANCELLED,353


## Stage 3 — Compare pickup-zone and time-block segments

Python summaries compare offer volumes, acceptance, contacted drivers, payments, completion, fares, and reports. Segments with at least 75 offers are used for the comparison of weaker-performing segments.

In [84]:
offers_table.columns

Index(['offer_id', 'cairo_time', 'year', 'month', 'day', 'weekday', 'hour',
       'passenger_id', 'driver_id', 'vehicle_id', 'vehicle_category',
       'pickup_zone_name', 'dropoff_zone_name', 'initial_fare_egp',
       'offer_status', 'ride_id', 'ride_status', 'final_fare_egp',
       'payment_attempts', 'ride_level_successful_payments',
       'total_refund_amount_egp', 'avg_passenger_to_driver_rating',
       'avg_driver_to_passenger_rating', 'total_reports'],
      dtype='object')

In [85]:
offers_table['time_block'] = offers_table['hour'].apply(
    lambda x: 'morning' if 6 <= x < 12 else
              'afternoon' if 12 <= x < 18 else
              'evening' if 18 <= x < 24 else
              'night')

In [86]:
offers_table.time_block.value_counts()

time_block
afternoon    3052
morning      2253
evening      1991
night         390
Name: count, dtype: int64

In [87]:
offers_table.pickup_zone_name.value_counts()

pickup_zone_name
Mansoura University         767
El Mashaya                  715
El Mokhtalat                673
El Gomhoria                 644
Talkha Center               575
Mansoura Railway Station    564
El Hosayneya                498
Talkha Old Market           494
Toriel                      474
Gedila                      452
Mit Khamis                  423
Sandoub                     401
Talkha Railway Station      352
El Rowda                    333
El Mohandessin              321
Name: count, dtype: int64

In [88]:
import warnings
warnings.filterwarnings("ignore")

grouped_table = {}
for pickup_zone in offers_table.pickup_zone_name.unique():
    grouped_table[pickup_zone] = {}
    for time_block in offers_table.time_block.unique():
        grouped_table[pickup_zone][time_block] = {'total_offers' : offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0]}
        grouped_table[pickup_zone][time_block]['total_accepted_offers'] = offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block) &
            (offers_table.offer_status == 'ACCEPTED')
        ].shape[0]
        grouped_table[pickup_zone][time_block]['accepted_offers_rate'] = round(offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block) &
            (offers_table.offer_status == 'ACCEPTED')
        ].shape[0] / offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] if offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] > 0 else 0, 2)
        grouped_table[pickup_zone][time_block]['distinct_drivers_recieving_offers'] = offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].driver_id.nunique()
        grouped_table[pickup_zone][time_block]['offers_per_contacted_driver'] = offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] / offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].driver_id.nunique() if offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].driver_id.nunique() > 0 else 0
        grouped_table[pickup_zone][time_block]['payment_secured_rides'] = offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][(offers_table.ride_status == 'COMPLETED') & (offers_table.ride_level_successful_payments > 0)].shape[0]
        grouped_table[pickup_zone][time_block]['payment_secured_rides_rate'] = round(offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][(offers_table.ride_status == 'COMPLETED') & (offers_table.ride_level_successful_payments > 0)].shape[0] / offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] if offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] > 0 else 0, 2)
        grouped_table[pickup_zone][time_block]['completed_rides'] = offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][offers_table.ride_status == 'COMPLETED'].shape[0]
        grouped_table[pickup_zone][time_block]['completion_rate_among_accepted_offers'] = round(offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][offers_table.ride_status == 'COMPLETED'].shape[0] / offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][offers_table.offer_status == 'ACCEPTED'].shape[0] if offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] > 0 else 0, 2)
        grouped_table[pickup_zone][time_block]['midian_initial_fare_egp'] = round(offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].initial_fare_egp.median() if offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] > 0 else 0, 2)
        grouped_table[pickup_zone][time_block]['report_rate_per_1,000_started_rides'] = round(offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][offers_table.total_reports > 0].shape[0] / offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ][offers_table.ride_status == 'COMPLETED'].shape[0] * 1000 if offers_table[
            (offers_table.pickup_zone_name == pickup_zone) &
            (offers_table.time_block == time_block)
        ].shape[0] > 0 else 0, 2)


# grouped_table

In [89]:
grouped_table_df = pd.DataFrame.from_dict({(i,j): grouped_table[i][j]
                           for i in grouped_table.keys()
                           for j in grouped_table[i].keys()},
                       orient='index')

grouped_table_df

total_offers  total_accepted_offers  \
Mansoura Railway Station night                38                     30   
                         morning             154                    120   
                         afternoon           222                    173   
                         evening             150                    111   
El Mokhtalat             night                32                     27   
                         morning             176                    132   
                         afternoon           271                    214   
                         evening             194                    145   
Mansoura University      night                37                     29   
                         morning             202                    146   
                         afternoon           312                    237   
                         evening             216                    170   
Talkha Center            night                29                     23   
                         morning             164                    130   
                         afternoon           231                    181   
                         evening             151                    104   
Talkha Old Market        night                33                     28   
                         morning             125                    100   
                         afternoon           206                    167   
                         evening             130                     91   
Talkha Railway Station   night                22                     16   
                         morning              88                     61   
                         afternoon           145                    110   
                         evening              97                     69   
Mit Khamis               night                25                     14   
                         morning             149                    116   
                         afternoon           159                    120   
                         evening              90                     74   
El Gomhoria              night                23                     13   
                         morning             176                    134   
                         afternoon           264                    197   
                         evening             181                    144   
Sandoub                  night                17                     15   
                         morning             119                     97   
                         afternoon           168                    128   
                         evening              97                     77   
Gedila                   night                24                     19   
                         morning             158                    115   
                         afternoon           163                    130   
                         evening             107                     81   
El Hosayneya             night                30                     21   
                         morning             175                    131   
                         afternoon           187                    141   
                         evening             106                     81   
Toriel                   night                19                     12   
                         morning             160                    122   
                         afternoon           179                    133   
                         evening             116                     94   
El Mashaya               night                24                     17   
                         morning             194                    154   
                         afternoon           292                    228   
                         evening             205                    141   
El Mohandessin           night                19             

### Lower-performing segments

Compare the lowest observed rates among segments meeting the offer-count threshold.

In [90]:
grouped_table_df.columns

Index(['total_offers', 'total_accepted_offers', 'accepted_offers_rate',
       'distinct_drivers_recieving_offers', 'offers_per_contacted_driver',
       'payment_secured_rides', 'payment_secured_rides_rate',
       'completed_rides', 'completion_rate_among_accepted_offers',
       'midian_initial_fare_egp', 'report_rate_per_1,000_started_rides'],
      dtype='object')

In [91]:
qualified_segments = grouped_table_df[
    (grouped_table_df.total_offers >= 75)
]

col_dct = {}

for col in ['accepted_offers_rate',
       'payment_secured_rides_rate',
       'completion_rate_among_accepted_offers']:
    print(f'For {col} the lowest perfomer is {qualified_segments[col].idxmin()}')
    col_dct[col] = qualified_segments[col].idxmin()

For accepted_offers_rate the lowest perfomer is ('Talkha Center', 'evening')
For payment_secured_rides_rate the lowest perfomer is ('El Mohandessin', 'morning')
For completion_rate_among_accepted_offers the lowest perfomer is ('El Mokhtalat', 'evening')


In [92]:
print(f'For report_rate_per_1,000_started_rides the lowest perfomer is {qualified_segments['report_rate_per_1,000_started_rides'].idxmax()}')
col_dct['report_rate_per_1,000_started_rides'] = qualified_segments['report_rate_per_1,000_started_rides'].idxmax()

For report_rate_per_1,000_started_rides the lowest perfomer is ('El Mokhtalat', 'evening')


In [93]:
qualified_segments.loc[col_dct.values()].drop_duplicates()

,,total_offers,total_accepted_offers,accepted_offers_rate,distinct_drivers_recieving_offers,offers_per_contacted_driver,payment_secured_rides,payment_secured_rides_rate,completed_rides,completion_rate_among_accepted_offers,midian_initial_fare_egp,"report_rate_per_1,000_started_rides"
Talkha Center,evening,151,104,0.69,81,1.864198,94,0.62,94,0.90,45.0,53.19
El Mohandessin,morning,95,67,0.71,61,1.557377,58,0.61,58,0.87,50.0,51.72
El Mokhtalat,evening,194,145,0.75,78,2.487179,125,0.64,125,0.86,50.0,88.00


Hmm, why do accepted rides in El Mokhtalat during the evening fail to complete or produce reports more often than the marketplace average?

## Stage 4 — Investigate El Mokhtalat evening

Compare ride outcomes for accepted offers in El Mokhtalat during the evening with the rest of the marketplace. Inspect failed rides by cancelling side, cancellation stage, and reason.

Contacted-driver counts do not measure all nearby available drivers.

In [94]:
accepted_offers_me_df = offers_table[
    (offers_table.offer_status == 'ACCEPTED') &
    (offers_table.pickup_zone_name == 'El Mokhtalat') &
    (offers_table.time_block == 'evening')
]

accepted_offers_me_df

,offer_id,cairo_time,year,month,day,weekday,hour,passenger_id,driver_id,vehicle_id,...,ride_id,ride_status,final_fare_egp,payment_attempts,ride_level_successful_payments,total_refund_amount_egp,avg_passenger_to_driver_rating,avg_driver_to_passenger_rating,total_reports,time_block
78,6393,2026-06-01 19:06:03,2026.0,6.0,1.0,1.0,19.0,248,1008,35,...,4825.0,CANCELLED_BEFORE_START,0.000000,1,0,0.0,NaN,NaN,0,evening
81,6396,2026-06-01 19:32:48,2026.0,6.0,1.0,1.0,19.0,346,1045,85,...,4828.0,COMPLETED,32.730000,1,1,0.0,3.8,NaN,0,evening
91,6406,2026-06-01 22:32:51,2026.0,6.0,1.0,1.0,22.0,277,1037,76,...,4837.0,CANCELLED_BEFORE_START,0.000000,1,0,0.0,NaN,NaN,1,evening
148,6463,2026-06-02 18:18:11,2026.0,6.0,2.0,2.0,18.0,535,1073,119,...,4880.0,COMPLETED,51.470001,1,1,0.0,4.8,3.25,0,evening
157,6472,2026-06-02 19:24:04,2026.0,6.0,2.0,2.0,19.0,296,1042,81,...,4887.0,COMPLETED,74.849998,1,1,0.0,4.2,NaN,0,evening
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7371,13686,2026-08-27 20:18:02,2026.0,8.0,27.0,4.0,20.0,791,1053,95,...,10376.0,CANCELLED_BEFORE_START,0.000000,1,0,0.0,NaN,NaN,0,evening
7496,13811,2026-08-28 22:48:22,2026.0,8.0,28.0,5.0,22.0,535,1067,112,...,10472.0,COMPLETED,43.970001,1,1,0.0,NaN,4.00,0,evening
7582,13897,2026-08-29 20:28:51,2026.0,8.0,29.0,6.0,20.0,394,1040,79,...,10536.0,COMPLETED,39.490002,1,1,0.0,4.6,NaN,0,evening
7585,13900,2026-08-29 20:56:37,2026.0,8.0,29.0,6.0,20.0,137,1071,117,...,10539.0,COMPLETED,52.540001,1,1,0.0,5.0,4.25,0,evening


In [95]:
accepted_offers_other_df = offers_table[
    (offers_table.offer_status == 'ACCEPTED') &
    ~((offers_table.pickup_zone_name == 'El Mokhtalat') & (offers_table.time_block == 'evening'))
]

accepted_offers_other_df

,offer_id,cairo_time,year,month,day,weekday,hour,passenger_id,driver_id,vehicle_id,...,ride_id,ride_status,final_fare_egp,payment_attempts,ride_level_successful_payments,total_refund_amount_egp,avg_passenger_to_driver_rating,avg_driver_to_passenger_rating,total_reports,time_block
0,6315,2026-06-01 03:15:52,2026.0,6.0,1.0,1.0,3.0,56,1048,89,...,4766.0,COMPLETED,61.349998,1,1,0.0,4.6,3.75,0,night
3,6318,2026-06-01 05:24:52,2026.0,6.0,1.0,1.0,5.0,439,1057,101,...,4767.0,COMPLETED,40.080002,1,1,0.0,4.4,5.00,0,night
4,6319,2026-06-01 06:01:39,2026.0,6.0,1.0,1.0,6.0,771,1015,45,...,4768.0,COMPLETED,45.230000,1,1,0.0,NaN,NaN,0,morning
5,6320,2026-06-01 06:18:04,2026.0,6.0,1.0,1.0,6.0,884,1026,60,...,4769.0,COMPLETED,62.310001,1,1,0.0,4.2,NaN,0,morning
7,6322,2026-06-01 06:40:30,2026.0,6.0,1.0,1.0,6.0,164,1025,59,...,4770.0,COMPLETED,62.020000,1,1,0.0,4.2,3.00,0,morning
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7678,13993,2026-08-30 19:43:09,2026.0,8.0,30.0,7.0,19.0,237,1011,39,...,10613.0,COMPLETED,57.770000,1,1,0.0,5.0,4.50,0,evening
7681,13996,2026-08-30 20:17:18,2026.0,8.0,30.0,7.0,20.0,115,1006,33,...,10614.0,COMPLETED,81.510002,1,1,0.0,4.6,5.00,0,evening
7682,13997,2026-08-30 20:23:07,2026.0,8.0,30.0,7.0,20.0,603,1008,35,...,10615.0,COMPLETED,58.880001,1,1,0.0,NaN,4.25,0,evening
7684,13999,2026-08-30 21:10:49,2026.0,8.0,30.0,7.0,21.0,720,1019,51,...,10616.0,CANCELLED_BEFORE_START,0.000000,1,0,0.0,NaN,NaN,1,evening


In [96]:
el_mokhtalat_ride_status = (
    accepted_offers_me_df.ride_status
    .value_counts(normalize=True)
    .round(3)
)

marketplace_ride_status = (
    accepted_offers_other_df.ride_status
    .value_counts(normalize=True)
    .round(3)
)

ride_status_comparison = pd.concat(
    [
        el_mokhtalat_ride_status,
        marketplace_ride_status
    ],
    axis=1
)

ride_status_comparison.columns = [
    'el_mokhtalat_evening',
    'rest_of_marketplace'
]

ride_status_comparison = ride_status_comparison.fillna(0)

ride_status_comparison

,el_mokhtalat_evening,rest_of_marketplace
ride_status,,
COMPLETED,0.862,0.920
CANCELLED_BEFORE_START,0.083,0.052
TERMINATED_EARLY,0.055,0.029
AWAITING_PAYMENT,0.000,0.000


In [97]:
analysis_grain_sql = """
WITH payment_summary AS (
    SELECT
        pa.ride_id,
        COUNT(*) AS payment_attempts,
        MAX(
            CASE
                WHEN pa.payment_status IN ('AUTHORIZED', 'CAPTURED') THEN 1
                ELSE 0
            END
        ) AS ride_level_successful_payments
    FROM payment_attempts AS pa
    GROUP BY pa.ride_id
),

refund_summary AS (
    SELECT
        pa.ride_id,
        SUM(rfnd.refund_amount_egp) FILTER (
            WHERE rfnd.refund_status = 'COMPLETED'
        ) AS total_refund_amount_egp
    FROM refunds AS rfnd
    JOIN payment_attempts AS pa ON rfnd.payment_attempt_id = pa.payment_attempt_id
    GROUP BY pa.ride_id
),

report_summary AS (
    SELECT
        rprt.ride_id,
        COUNT(rprt.report_id) AS total_reports
    FROM reports AS rprt
    GROUP BY rprt.ride_id
)

SELECT
    o.offer_id,
    o.initiated_at AT TIME ZONE 'Africa/Cairo' AS cairo_time,
    EXTRACT(YEAR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS year,
    EXTRACT(MONTH FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS month,
    EXTRACT(DAY FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS day,
    EXTRACT(ISODOW FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS weekday,
    EXTRACT(HOUR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo') AS hour,
    o.passenger_id,
    o.driver_id,
    o.vehicle_id,
    v.vehicle_category,
    zp.zone_name AS pickup_zone_name,
    zd.zone_name AS dropoff_zone_name,
    o.initial_fare_egp,
    o.offer_status,
    r.ride_id,
    r.ride_status,
    r.final_fare_egp,
    r.cancelled_by,
    r.cancellation_stage,
    r.cancellation_reason,
    COALESCE(ps.payment_attempts, 0) AS payment_attempts,
    COALESCE(ps.ride_level_successful_payments, 0) AS ride_level_successful_payments,
    COALESCE(rs.total_refund_amount_egp, 0) AS total_refund_amount_egp,
    (
        pdr.attitude_score
        + pdr.driving_safety_score
        + pdr.vehicle_cleanliness_score
        + pdr.comfort_score
        + pdr.route_quality_score
    ) / 5.0 AS avg_passenger_to_driver_rating,
    (
        dpr.attitude_score
        + dpr.punctuality_score
        + dpr.pickup_cooperation_score
        + dpr.respect_safety_score
    ) / 4.0 AS avg_driver_to_passenger_rating,
    COALESCE(rps.total_reports, 0) AS total_reports
FROM offers AS o
LEFT JOIN rides AS r ON o.offer_id = r.offer_id
LEFT JOIN vehicles AS v ON o.vehicle_id = v.vehicle_id
LEFT JOIN zones AS zp ON o.pickup_zone_id = zp.zone_id
LEFT JOIN zones AS zd ON o.dropoff_zone_id = zd.zone_id
LEFT JOIN payment_summary AS ps ON r.ride_id = ps.ride_id
LEFT JOIN refund_summary AS rs ON r.ride_id = rs.ride_id
LEFT JOIN passenger_driver_ratings AS pdr ON r.ride_id = pdr.ride_id
LEFT JOIN driver_passenger_ratings AS dpr ON r.ride_id = dpr.ride_id
LEFT JOIN report_summary AS rps ON r.ride_id = rps.ride_id
WHERE o.initiated_at >= :analysis_start
    AND o.initiated_at < :analysis_end
ORDER BY o.initiated_at
"""


offers_table_plus = run_sql(analysis_grain_sql, analysis_params)
offers_table_plus['time_block'] = offers_table_plus['hour'].apply(
    lambda x: 'morning' if 6 <= x < 12 else
              'afternoon' if 12 <= x < 18 else
              'evening' if 18 <= x < 24 else
              'night')
offers_table_plus

,offer_id,cairo_time,year,month,day,weekday,hour,passenger_id,driver_id,vehicle_id,...,cancelled_by,cancellation_stage,cancellation_reason,payment_attempts,ride_level_successful_payments,total_refund_amount_egp,avg_passenger_to_driver_rating,avg_driver_to_passenger_rating,total_reports,time_block
0,6315,2026-06-01 03:15:52,2026.0,6.0,1.0,1.0,3.0,56,1048,89,...,NaN,NaN,NaN,1,1,0.0,4.6,3.75,0,night
1,6316,2026-06-01 03:19:39,2026.0,6.0,1.0,1.0,3.0,913,1033,70,...,NaN,NaN,NaN,0,0,0.0,NaN,NaN,0,night
2,6317,2026-06-01 04:37:46,2026.0,6.0,1.0,1.0,4.0,794,1026,60,...,NaN,NaN,NaN,0,0,0.0,NaN,NaN,0,night
3,6318,2026-06-01 05:24:52,2026.0,6.0,1.0,1.0,5.0,439,1057,101,...,NaN,NaN,NaN,1,1,0.0,4.4,5.00,0,night
4,6319,2026-06-01 06:01:39,2026.0,6.0,1.0,1.0,6.0,771,1015,45,...,NaN,NaN,NaN,1,1,0.0,NaN,NaN,0,morning
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7681,13996,2026-08-30 20:17:18,2026.0,8.0,30.0,7.0,20.0,115,1006,33,...,NaN,NaN,NaN,1,1,0.0,4.6,5.00,0,evening
7682,13997,2026-08-30 20:23:07,2026.0,8.0,30.0,7.0,20.0,603,1008,35,...,NaN,NaN,NaN,1,1,0.0,NaN,4.25,0,evening
7683,13998,2026-08-30 20:58:20,2026.0,8.0,30.0,7.0,20.0,557,1068,113,...,NaN,NaN,NaN,0,0,0.0,NaN,NaN,0,evening
7684,13999,2026-08-30 21:10:49,2026.0,8.0,30.0,7.0,21.0,720,1019,51,...,DRIVER,BEFORE_DRIVER_ARRIVAL,UNSAFE_PICKUP,1,0,0.0,NaN,NaN,1,evening


In [98]:
accepted_offers_me_df = offers_table_plus[
    (offers_table_plus.offer_status == 'ACCEPTED') &
    (offers_table_plus.pickup_zone_name == 'El Mokhtalat') &
    (offers_table_plus.time_block == 'evening')
]

accepted_offers_me_df

,offer_id,cairo_time,year,month,day,weekday,hour,passenger_id,driver_id,vehicle_id,...,cancelled_by,cancellation_stage,cancellation_reason,payment_attempts,ride_level_successful_payments,total_refund_amount_egp,avg_passenger_to_driver_rating,avg_driver_to_passenger_rating,total_reports,time_block
78,6393,2026-06-01 19:06:03,2026.0,6.0,1.0,1.0,19.0,248,1008,35,...,PASSENGER,BEFORE_DRIVER_ARRIVAL,CHANGE_OF_PLANS,1,0,0.0,NaN,NaN,0,evening
81,6396,2026-06-01 19:32:48,2026.0,6.0,1.0,1.0,19.0,346,1045,85,...,NaN,NaN,NaN,1,1,0.0,3.8,NaN,0,evening
91,6406,2026-06-01 22:32:51,2026.0,6.0,1.0,1.0,22.0,277,1037,76,...,DRIVER,BEFORE_DRIVER_ARRIVAL,PASSENGER_NO_SHOW,1,0,0.0,NaN,NaN,1,evening
148,6463,2026-06-02 18:18:11,2026.0,6.0,2.0,2.0,18.0,535,1073,119,...,NaN,NaN,NaN,1,1,0.0,4.8,3.25,0,evening
157,6472,2026-06-02 19:24:04,2026.0,6.0,2.0,2.0,19.0,296,1042,81,...,NaN,NaN,NaN,1,1,0.0,4.2,NaN,0,evening
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7371,13686,2026-08-27 20:18:02,2026.0,8.0,27.0,4.0,20.0,791,1053,95,...,DRIVER,BEFORE_DRIVER_ARRIVAL,UNSAFE_PICKUP,1,0,0.0,NaN,NaN,0,evening
7496,13811,2026-08-28 22:48:22,2026.0,8.0,28.0,5.0,22.0,535,1067,112,...,NaN,NaN,NaN,1,1,0.0,NaN,4.00,0,evening
7582,13897,2026-08-29 20:28:51,2026.0,8.0,29.0,6.0,20.0,394,1040,79,...,NaN,NaN,NaN,1,1,0.0,4.6,NaN,0,evening
7585,13900,2026-08-29 20:56:37,2026.0,8.0,29.0,6.0,20.0,137,1071,117,...,NaN,NaN,NaN,1,1,0.0,5.0,4.25,0,evening


In [99]:
el_mokhtalat_failed_rides = accepted_offers_me_df[
    accepted_offers_me_df.ride_status.isin([
        'CANCELLED_BEFORE_START',
        'TERMINATED_EARLY'
    ])
]

el_mokhtalat_failed_rides.groupby(
    [
        'ride_status',
        'cancelled_by',
        'cancellation_stage',
        'cancellation_reason'
    ],
    dropna=False,
    observed=True
).size().sort_values(ascending=False).to_frame('count')

count
ride_status            cancelled_by cancellation_stage    cancellation_reason       
CANCELLED_BEFORE_START DRIVER       BEFORE_DRIVER_ARRIVAL EMERGENCY                2
                                                          PASSENGER_NO_SHOW        2
                                                          UNSAFE_PICKUP            2
                       PASSENGER    BEFORE_DRIVER_ARRIVAL CHANGE_OF_PLANS          2
TERMINATED_EARLY       PASSENGER    DURING_RIDE           EMERGENCY                2
CANCELLED_BEFORE_START DRIVER       BEFORE_DRIVER_ARRIVAL VEHICLE_PROBLEM          1
                       PASSENGER    AFTER_DRIVER_ARRIVAL  CHANGE_OF_PLANS          1
                                                          WAIT_TOO_LONG            1
                                    BEFORE_DRIVER_ARRIVAL WRONG_PICKUP             1
TERMINATED_EARLY       DRIVER       DURING_RIDE           DRIVER_BEHAVIOR          1
                                                          EMERGENCY                1
                                                          PASSENGER_BEHAVIOR       1
                       PASSENGER    DURING_RIDE           SAFETY_CONCERN           1
                                                          VEHICLE_PROBLEM          1
                       SYSTEM       DURING_RIDE           VEHICLE_PROBLEM          1

El Mokhtalat evening has unusually high failure rates, with pre-start cancellations occurring more often from the driver side, but the sample does not reveal one dominant cancellation reason.

## Stage 5 — Ratings and reports

Summarize started rides, rating coverage, average ratings, and reported-ride rates across the marketplace and by pickup zone and time block. Segment quality comparisons use a minimum of 30 started rides.

Missing ratings are kept separate from low ratings.

In [100]:
started_rides_table = offers_table[
    offers_table.ride_status.isin([
        'IN_PROGRESS',
        'COMPLETED',
        'TERMINATED_EARLY'
    ])
]

In [101]:
quality_results = {}

quality_results['started_rides'] = (
    started_rides_table.shape[0]
)

quality_results['passenger_rating_coverage'] = round(
    started_rides_table[
        'avg_passenger_to_driver_rating'
    ].notna().mean(),
    2
)

quality_results['avg_passenger_to_driver_rating'] = round(
    started_rides_table[
        'avg_passenger_to_driver_rating'
    ].mean(),
    2
)

quality_results['driver_rating_coverage'] = round(
    started_rides_table[
        'avg_driver_to_passenger_rating'
    ].notna().mean(),
    2
)

quality_results['avg_driver_to_passenger_rating'] = round(
    started_rides_table[
        'avg_driver_to_passenger_rating'
    ].mean(),
    2
)

quality_results['reported_rides'] = (
    started_rides_table.total_reports
    .gt(0)
    .sum()
)

quality_results['report_rate_per_1000_started_rides'] = round(
    quality_results['reported_rides']
    / quality_results['started_rides']
    * 1000,
    2
)

quality_results

{'started_rides': 5545,
 'passenger_rating_coverage': 0.7,
 'avg_passenger_to_driver_rating': 4.23,
 'driver_rating_coverage': 0.54,
 'avg_driver_to_passenger_rating': 4.2,
 'reported_rides': 167,
 'report_rate_per_1000_started_rides': 30.12}

Interpretation:
- 5,545 rides started.
- Passengers rated drivers after 70% of those rides.
- Drivers rated passengers after only 54%.
- Submitted ratings were generally high and similar: 4.23 versus 4.20.
- 167 started rides received at least one report.
- That equals 30.12 reported rides per 1,000 started rides, or about 3.01%.

In [102]:
import warnings
warnings.filterwarnings("ignore")

grouped_quality_table = {}
for pickup_zone in started_rides_table.pickup_zone_name.unique():
    grouped_quality_table[pickup_zone] = {}
    for time_block in started_rides_table.time_block.unique():
        grouped_quality_table[pickup_zone][time_block] = {
            'started_rides' : started_rides_table[
                (started_rides_table.pickup_zone_name == pickup_zone) &
                (started_rides_table.time_block == time_block)
            ].shape[0]
        }
        grouped_quality_table[pickup_zone][time_block]['passenger_rating_coverage'] = round(
            started_rides_table[
                (started_rides_table.pickup_zone_name == pickup_zone) &
                (started_rides_table.time_block == time_block)
            ]['avg_passenger_to_driver_rating'].notna().mean(),
            2
        )
        grouped_quality_table[pickup_zone][time_block]['avg_passenger_to_driver_rating'] = round(
            started_rides_table[
                (started_rides_table.pickup_zone_name == pickup_zone) &
                (started_rides_table.time_block == time_block)
            ]['avg_passenger_to_driver_rating'].mean(),
            2
        )
        grouped_quality_table[pickup_zone][time_block]['driver_rating_coverage'] = round(
            started_rides_table[
                (started_rides_table.pickup_zone_name == pickup_zone) &
                (started_rides_table.time_block == time_block)
            ]['avg_driver_to_passenger_rating'].notna().mean(),
            2
        )
        grouped_quality_table[pickup_zone][time_block]['avg_driver_to_passenger_rating'] = round(
            started_rides_table[
                (started_rides_table.pickup_zone_name == pickup_zone) &
                (started_rides_table.time_block == time_block)
            ]['avg_driver_to_passenger_rating'].mean(),
            2
        )
        grouped_quality_table[pickup_zone][time_block]['reported_rides'] = (
            started_rides_table[
                (started_rides_table.pickup_zone_name == pickup_zone) &
                (started_rides_table.time_block == time_block)
            ].total_reports.gt(0).sum()
        )
        grouped_quality_table[pickup_zone][time_block]['report_rate_per_1000_started_rides'] = round(
            grouped_quality_table[pickup_zone][time_block]['reported_rides']
            / grouped_quality_table[pickup_zone][time_block]['started_rides']
            * 1000,
            2
        )


grouped_quality_table_df = pd.DataFrame.from_dict({(i,j): grouped_quality_table[i][j]
                           for i in grouped_quality_table.keys()
                           for j in grouped_quality_table[i].keys()},
                       orient='index')

grouped_quality_table_df.index.names = [
    'pickup_zone_name',
    'time_block'
]

qualified_quality_segments = grouped_quality_table_df[
    grouped_quality_table_df.started_rides >= 30
]

qualified_quality_segments

started_rides  passenger_rating_coverage  \
pickup_zone_name         time_block                                             
Mansoura Railway Station night                  30                       0.70   
                         morning               116                       0.66   
                         afternoon             168                       0.69   
                         evening               108                       0.68   
Talkha Center            morning               123                       0.72   
                         afternoon             170                       0.65   
                         evening                98                       0.63   
Mansoura University      morning               142                       0.70   
                         afternoon             217                       0.71   
                         evening               161                       0.68   
Talkha Old Market        morning                95                       0.67   
                         afternoon             153                       0.75   
                         evening                87                       0.64   
Talkha Railway Station   morning                59                       0.59   
                         afternoon             108                       0.63   
                         evening                63                       0.76   
Mit Khamis               morning               110                       0.66   
                         afternoon             113                       0.69   
                         evening                69                       0.70   
Sandoub                  morning                92                       0.67   
                         afternoon             117                       0.62   
                         evening                74                       0.73   
El Gomhoria              morning               128                       0.69   
                         afternoon             187                       0.68   
                         evening               130                       0.79   
Gedila                   morning               110                       0.73   
                         afternoon             121                       0.76   
                         evening                78                       0.63   
El Hosayneya             morning               125                       0.81   
                         afternoon             132                       0.69   
                         evening                79                       0.78   
Toriel                   morning               118                       0.78   
                         afternoon             130                       0.84   
                         evening                86                       0.76   
El Mokhtalat             morning               126                       0.68   
                         afternoon             201                       0.64   
                         evening               133                       0.72   
El Mohandessin           morning                62                       0.61   
                         afternoon              90                       0.70   
                         evening                57                       0.63   
El Mashaya               morning               149                       0.64   
                         afternoon             218                       0.67   
                         evening               136                       0.70   
El Rowda                 morning                83                       0.64   
                         afternoon              91                       0.70   
                         evening                50                       0.70   

                                     avg_passenger_to_driver_rating  \
pickup_zone_name         time_block                                   
Mansou

In [103]:
# conditional formatting for the quality metrics
# coloring based on coparision to the rest of the marketplace (quality_results)
from IPython.display import display, HTML

qualified_quality_segments_styled = qualified_quality_segments.style.applymap(
    lambda x: 'background-color: red' if x < quality_results['avg_passenger_to_driver_rating'] else 'background-color: green' if x > quality_results['avg_passenger_to_driver_rating'] else '',
    subset=['avg_passenger_to_driver_rating']).applymap(
    lambda x: 'background-color: red' if x < quality_results['avg_driver_to_passenger_rating'] else 'background-color: green' if x > quality_results['avg_driver_to_passenger_rating'] else '',
    subset=['avg_driver_to_passenger_rating']).applymap(
    lambda x: 'background-color: red' if x > quality_results['report_rate_per_1000_started_rides'] else 'background-color: green' if x < quality_results['report_rate_per_1000_started_rides'] else '',
    subset=['report_rate_per_1000_started_rides']).applymap(
    lambda x: 'background-color: red' if x < quality_results['passenger_rating_coverage'] else 'background-color: green' if x > quality_results['passenger_rating_coverage'] else '',
    subset=['passenger_rating_coverage']).applymap(
    lambda x: 'background-color: red' if x < quality_results['driver_rating_coverage'] else 'background-color: green' if x > quality_results['driver_rating_coverage'] else '',
    subset=['driver_rating_coverage'])

qualified_quality_segments_styled

1. Ratings are generally high
Marketplace averages:
Passenger → driver: 4.23
Driver → passenger: 4.20
Most segment averages remain near those values. The differences are generally small, so red cells such as 4.21 versus 4.23 are not meaningful by themselves.
The lowest passenger-to-driver averages are:
- Gedila + afternoon: 4.11
- El Rowda + afternoon: 4.11
- Mit Khamis + evening: 4.15
These deserve investigation, but none represents disastrous service.
The lowest driver-to-passenger averages include:
- Mansoura Railway Station + night: 4.02, but only around 15 submitted driver ratings.
- Talkha Center + morning: 4.05
- Sandoub + morning: 4.10
2. Rating coverage varies considerably
Passenger coverage ranges from roughly 59% to 84%.
Driver coverage is generally lower and reaches:
- Mit Khamis + morning: 45%
- El Mokhtalat + afternoon: 46%
- El Mohandessin + evening: 46%
That is a data-completeness or participation problem, not proof of bad service. Missing ratings are not bad ratings.
3. Reports show the strongest risk signal
The marketplace baseline is:
30.12 reported rides per 1,000 started rides
Notable qualified segments:
Segment	Reported rides	Started rides	Rate
Mansoura Railway Station + night	3	30	100.00
El Mokhtalat + evening	8	133	60.15
Toriel + morning	7	118	59.32
Mit Khamis + evening	4	69	57.97
El Mokhtalat + morning	7	126	55.56


Mansoura Railway Station’s night rate is the highest, but 3/30 is unstable. One additional report would move the rate massively.
El Mokhtalat evening and Toriel morning are more credible warning signals because they have larger samples and around twice the marketplace report rate.
4. Ratings and reports do not tell the same story
El Mokhtalat evening has:
Passenger rating: 4.21
Driver rating:    4.19
Report rate:      60.15
Its average ratings look normal, but its report rate is high. Ratings and reports capture different signals; report severity and validity require reviewing individual cases.
Gedila afternoon shows another interesting disagreement:
Passenger → driver: 4.11
Driver → passenger: 4.25
Passengers appear less satisfied than drivers in that segment. El Rowda afternoon shows a similar difference.
Summary
Overall ratings remain high across the marketplace, with limited differences between most segments. The clearer quality concern comes from reports rather than average ratings. El Mokhtalat evening and Toriel morning have report rates close to twice the marketplace baseline while maintaining reasonable sample sizes. Rating coverage, particularly from drivers, is inconsistent and limits confidence in some segment comparisons.

## Stage 6 — Investigation extract

A parameterized PostgreSQL query selects up to 200 rides from El Mokhtalat during the evening for review. It includes route and outcome details, payment summaries, cancellations, ratings, and report information.

Reported rides and unsuccessful outcomes are ordered first. The row limit is applied in PostgreSQL.

In [104]:
analysis_params.update(
    {
        "selected_zone": "El Mokhtalat",
        "selected_time_block": "evening",
    }
)

investigation_extract_sql = """
WITH payment_summary AS (
    SELECT
        pa.ride_id,
        COUNT(*) AS payment_attempts,
        COUNT(*) FILTER (
            WHERE pa.payment_status = 'FAILED'
        ) AS failed_payment_attempts,
        BOOL_OR(
            pa.payment_status IN ('AUTHORIZED', 'CAPTURED')
        ) AS payment_secured,
        STRING_AGG(
            DISTINCT pa.failure_reason,
            ' | '
        ) FILTER (
            WHERE pa.failure_reason IS NOT NULL
        ) AS payment_failure_reasons
    FROM payment_attempts AS pa
    GROUP BY pa.ride_id
),

report_summary AS (
    SELECT
        rpt.ride_id,
        COUNT(*) AS total_reports,
        COUNT(*) FILTER (
            WHERE rpt.report_status IN ('OPEN', 'IN_PROGRESS')
        ) AS unresolved_reports,
        STRING_AGG(
            DISTINCT rpt.report_category,
            ', '
        ) AS report_categories,
        STRING_AGG(
            DISTINCT rpt.report_status,
            ', '
        ) AS report_statuses,
        STRING_AGG(
            DISTINCT rpt.description,
            ' | '
        ) AS report_notes
    FROM reports AS rpt
    GROUP BY rpt.ride_id
)

SELECT
    r.ride_id,
    o.offer_id,
    o.passenger_id,
    o.driver_id,
    o.vehicle_id,
    o.initiated_at AT TIME ZONE 'Africa/Cairo' AS cairo_time,
    z.zone_name AS pickup_zone_name,
    z2.zone_name AS dropoff_zone_name,
    o.offer_status,
    r.ride_status,
    r.final_fare_egp,
    COALESCE(ps.payment_attempts, 0) AS payment_attempts,
    COALESCE(ps.failed_payment_attempts, 0) AS failed_payment_attempts,
    COALESCE(ps.payment_secured, FALSE) AS payment_secured,
    ps.payment_failure_reasons,
    r.cancelled_by,
    r.cancellation_stage,
    r.cancellation_reason,

    ROUND((
        SELECT AVG(score)
        FROM UNNEST(ARRAY[
            pdr.attitude_score,
            pdr.driving_safety_score,
            pdr.vehicle_cleanliness_score,
            pdr.comfort_score,
            pdr.route_quality_score
        ]) AS score
    ), 2) AS avg_passenger_to_driver_rating,

    ROUND((
        SELECT AVG(score)
        FROM UNNEST(ARRAY[
            dpr.attitude_score,
            dpr.punctuality_score,
            dpr.pickup_cooperation_score,
            dpr.respect_safety_score
        ]) AS score
    ), 2) AS avg_driver_to_passenger_rating,

    COALESCE(rps.total_reports, 0) AS total_reports,
    COALESCE(rps.unresolved_reports, 0) AS unresolved_reports,
    rps.report_categories,
    rps.report_statuses,
    rps.report_notes

FROM offers AS o
JOIN rides AS r ON o.offer_id = r.offer_id
LEFT JOIN zones AS z ON o.pickup_zone_id = z.zone_id
LEFT JOIN zones AS z2 ON o.dropoff_zone_id = z2.zone_id
LEFT JOIN payment_summary AS ps ON r.ride_id = ps.ride_id
LEFT JOIN passenger_driver_ratings AS pdr ON r.ride_id = pdr.ride_id
LEFT JOIN driver_passenger_ratings AS dpr ON r.ride_id = dpr.ride_id
LEFT JOIN report_summary AS rps ON r.ride_id = rps.ride_id

WHERE o.initiated_at >= :analysis_start
    AND o.initiated_at < :analysis_end
    AND z.zone_name = :selected_zone
    AND CASE
        WHEN EXTRACT(
            HOUR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo'
        ) BETWEEN 6 AND 11 THEN 'morning'

        WHEN EXTRACT(
            HOUR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo'
        ) BETWEEN 12 AND 17 THEN 'afternoon'

        WHEN EXTRACT(
            HOUR FROM o.initiated_at AT TIME ZONE 'Africa/Cairo'
        ) BETWEEN 18 AND 23 THEN 'evening'

        ELSE 'night'
    END = :selected_time_block

ORDER BY
    CASE
        WHEN COALESCE(rps.total_reports, 0) > 0 THEN 1
        WHEN r.ride_status = 'TERMINATED_EARLY' THEN 2
        WHEN r.ride_status = 'CANCELLED_BEFORE_START' THEN 3
        WHEN COALESCE(ps.payment_secured, FALSE) = FALSE THEN 4
        ELSE 5
    END,
    o.initiated_at DESC

LIMIT 200
"""

investigation_extract = run_sql(
    investigation_extract_sql,
    analysis_params
)

investigation_extract

,ride_id,offer_id,passenger_id,driver_id,vehicle_id,cairo_time,pickup_zone_name,dropoff_zone_name,offer_status,ride_status,...,cancelled_by,cancellation_stage,cancellation_reason,avg_passenger_to_driver_rating,avg_driver_to_passenger_rating,total_reports,unresolved_reports,report_categories,report_statuses,report_notes
0,9922,13082,437,1017,49,2026-08-20 22:25:59,El Mokhtalat,Mit Khamis,ACCEPTED,TERMINATED_EARLY,...,PASSENGER,DURING_RIDE,EMERGENCY,NaN,NaN,1,0,VEHICLE_ISSUE,REJECTED,A vehicle condition issue was reported.
1,9775,12897,4,1013,42,2026-08-18 19:15:13,El Mokhtalat,Talkha Old Market,ACCEPTED,TERMINATED_EARLY,...,DRIVER,DURING_RIDE,DRIVER_BEHAVIOR,NaN,NaN,1,0,PASSENGER_BEHAVIOR,RESOLVED,Passenger conduct affected the trip.
2,8879,11741,604,1080,127,2026-08-05 19:23:39,El Mokhtalat,El Mashaya,ACCEPTED,CANCELLED_BEFORE_START,...,PASSENGER,BEFORE_DRIVER_ARRIVAL,WRONG_PICKUP,NaN,NaN,1,0,PASSENGER_BEHAVIOR,RESOLVED,Passenger conduct affected the trip.
3,8407,11098,294,989,11,2026-07-28 21:00:25,El Mokhtalat,El Gomhoria,ACCEPTED,TERMINATED_EARLY,...,PASSENGER,DURING_RIDE,VEHICLE_PROBLEM,NaN,NaN,1,1,PAYMENT_ISSUE,IN_PROGRESS,The charged amount or payment handling was dis...
4,7943,10501,988,1044,84,2026-07-21 19:59:23,El Mokhtalat,El Rowda,ACCEPTED,TERMINATED_EARLY,...,DRIVER,DURING_RIDE,PASSENGER_BEHAVIOR,3.2,4.00,1,0,DRIVER_BEHAVIOR,REJECTED,Driver conduct requires review.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,4890,6476,823,1011,39,2026-06-02 19:42:14,El Mokhtalat,Talkha Railway Station,ACCEPTED,COMPLETED,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
141,4889,6475,461,1008,35,2026-06-02 19:33:28,El Mokhtalat,El Rowda,ACCEPTED,COMPLETED,...,NaN,NaN,NaN,NaN,4.50,0,0,NaN,NaN,NaN
142,4887,6472,296,1042,81,2026-06-02 19:24:04,El Mokhtalat,Gedila,ACCEPTED,COMPLETED,...,NaN,NaN,NaN,4.2,NaN,0,0,NaN,NaN,NaN
143,4880,6463,535,1073,119,2026-06-02 18:18:11,El Mokhtalat,Mansoura Railway Station,ACCEPTED,COMPLETED,...,NaN,NaN,NaN,4.8,3.25,0,0,NaN,NaN,NaN
